# Explorador interactivo de impacto raqueta-bola

Este cuaderno usa `table_tennis_simulation.py` para variar parámetros del golpe con sliders, simular la trayectoria resultante e identificar los momentos 1-6 de entrenamiento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

from table_tennis_simulation import (
    RacketImpactParameters, TABLE_HEIGHT, BALL_RADIUS, TABLE_LENGTH, TABLE_WIDTH,
    simulate_racket_impact, identify_trajectory_moments, draw_racket, plot_table
)


In [ ]:
def summarize_moments(moments):
    for number, moment in moments.items():
        interval = f", intervalo={moment.interval}, punto medio={np.round(moment.midpoint, 1)}" if moment.interval else ""
        print(f"Momento {number}: {moment.name}; t={moment.time:.3f}s; punto={np.round(moment.point, 1)}{interval}")


def explore(ball_vx=-2000, ball_vy=0, ball_vz=-1000,
            omega_x=0, omega_y=0, omega_z=471,
            friction=0.60, restitution=0.85,
            angle_x=0, angle_y=-30, angle_z=0,
            racket_vx=2000, racket_vy=0, racket_vz=1000):
    params = RacketImpactParameters(
        ball_velocity=(ball_vx, ball_vy, ball_vz),
        ball_omega=(omega_x, omega_y, omega_z),
        rubber_friction=friction,
        rubber_restitution=restitution,
        racket_angle=(angle_x, angle_y, angle_z),
        racket_velocity=(racket_vx, racket_vy, racket_vz),
    )
    result = simulate_racket_impact(params, dt=0.005, t_max=2.0)
    moments = identify_trajectory_moments(result)

    fig = plt.figure(figsize=(12, 5))
    ax = fig.add_subplot(121, projection='3d')
    ax.plot(result.x[0], result.x[1], result.x[2], color='purple')
    draw_racket(ax, center=params.ball_position, angle=params.racket_angle)
    ax.set_xlim(-200, TABLE_LENGTH + 600)
    ax.set_ylim(-300, TABLE_WIDTH + 300)
    ax.set_zlim(0, 1800)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_zlabel('Z (mm)')
    ax.set_title('Trayectoria y raqueta')

    ax2 = fig.add_subplot(122)
    t = result.t
    ax2.plot(t, result.x[2], label='altura bola')
    ax2.axhline(TABLE_HEIGHT + BALL_RADIUS, color='tab:gray', linestyle='--', label='nivel mesa + radio')
    for number, moment in moments.items():
        ax2.scatter(moment.time, moment.point[2], label=f'M{number}')
    ax2.set_xlabel('Tiempo (s)')
    ax2.set_ylabel('Z (mm)')
    ax2.legend()
    plt.tight_layout()
    summarize_moments(moments)
    return result, moments


In [ ]:
interact(
    explore,
    ball_vx=FloatSlider(value=-2000, min=-12000, max=12000, step=250, description='Bola Vx'),
    ball_vy=FloatSlider(value=0, min=-8000, max=8000, step=250, description='Bola Vy'),
    ball_vz=FloatSlider(value=-1000, min=-8000, max=8000, step=250, description='Bola Vz'),
    omega_x=FloatSlider(value=0, min=-1000, max=1000, step=25, description='Ωx'),
    omega_y=FloatSlider(value=0, min=-1000, max=1000, step=25, description='Ωy'),
    omega_z=FloatSlider(value=471, min=-1500, max=1500, step=25, description='Ωz'),
    friction=FloatSlider(value=0.60, min=0, max=1.5, step=0.05, description='Fricción'),
    restitution=FloatSlider(value=0.85, min=0, max=1.2, step=0.05, description='Restitución'),
    angle_x=FloatSlider(value=0, min=-90, max=90, step=2.5, description='Ángulo X'),
    angle_y=FloatSlider(value=-30, min=-90, max=90, step=2.5, description='Ángulo Y'),
    angle_z=FloatSlider(value=0, min=-180, max=180, step=5, description='Ángulo Z'),
    racket_vx=FloatSlider(value=2000, min=-8000, max=12000, step=250, description='Raqueta Vx'),
    racket_vy=FloatSlider(value=0, min=-8000, max=8000, step=250, description='Raqueta Vy'),
    racket_vz=FloatSlider(value=1000, min=-8000, max=8000, step=250, description='Raqueta Vz'),
);
